# CS 580 | 3-1 Activity | Optimizer and Rate Schedules


## Instructions

<div class="alert alert-block alert-info" style="color:black;">
Complete the TODOs in each section below. Refer to the activity guidelines and rubric for requirements. Your final submission should include three types of evidence:

1. **Working code**  
   Include the code you wrote or modified to complete each TODO. The code should show what you tested and should run without errors.

2. **Saved output**  
   Include evidence of your working code and results, such as printed output, tables, or plots generated from your code. This output should show what happened when you ran your code and what results it produced.

3. **Brief annotations**  
   Add concise explanations in Markdown cells or code comments. Explain why you made each important choice and what the results show.

Use the rubric criteria as the minimum requirements for each proficiency level. Strong submissions do more than run code. They compare results, explain patterns, connect the output to model behavior, support claims with concrete evidence, and cite borrowed ideas or outside sources using APA 7 style.

</div>


## 0. Setup

<div class="alert alert-block alert-warning" style="color:black;">

**Package setup note:** Run the next cell only if Python reports a missing package.

The command installs the activity requirements into the current notebook kernel environment. After installation, restart the kernel and run all cells again using the notebook menu.

After installation, restart the kernel. Then run all cells from the beginning of the notebook.

</div>


In [1]:
%pip install -r requirements.txt


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.8 MB/s  0:00:00m 9.2 MB/s eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [68]:
!df -h

Filesystem       Size  Used Avail Use% Mounted on
/dev/root         29G   28G   43M 100% /
tmpfs            7.8G   91M  7.7G   2% /dev/shm
tmpfs            3.1G  1.1M  3.1G   1% /run
tmpfs            5.0M     0  5.0M   0% /run/lock
efivarfs         128K  3.8K  120K   4% /sys/firmware/efi/efivars
/dev/nvme0n1p16  881M  178M  642M  22% /boot
/dev/nvme0n1p15  105M  6.2M   99M   6% /boot/efi
tmpfs            1.6G  108K  1.6G   1% /run/user/1000


### 0.1 Import Packages

In [1]:
# Import Python standard library modules.
import itertools
import os
import tempfile

import os
import urllib.request
import time
import warnings

# Import common third-party packages.
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Import deep learning packages.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import InverseTimeDecay, CosineDecay



2026-10-09 13:01:07.951855: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ubuntu/.local/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### 0.2 Configure Environment

In [2]:
# Development controls
DEV_MODE = True  # TODO: Set to False before saving final version of notebook.

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

### 0.3 Check Environment

In [3]:
if DEV_MODE:
    print("Running in development mode with limited runtime and resource usage.")
    print("Reminder: Set DEV_MODE = False before final training.\n")

else:
    print("Running in full mode with no limits on runtime or resource usage.")
    warnings.filterwarnings("ignore", category=UserWarning)

print(f"TensorFlow version: {tf.__version__}")
print("Available physical devices:", tf.config.list_physical_devices())

Running in development mode with limited runtime and resource usage.
Reminder: Set DEV_MODE = False before final training.

TensorFlow version: 2.20.0
Available physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-10-09 13:01:10.721760: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 1. Problem

### 1.1. Task Statement

In this notebook, I fine-tune a pre-trained computer vision model to classify images from **[CIFAR-10 or Fashion MNIST]**. The goal is not only to build an image classifier, but also to study how different optimization choices affect model training and performance.

The input to the model is a set of labeled images from the selected dataset. Each image belongs to one predefined class, and the model learns to predict the correct class label for new images. Because I am using transfer learning, I start with a model that has already learned general image features and then adapt it to this smaller classification task.

The main focus of this activity is tuning the fine-tuning process. I configure and compare multiple optimizer and learning-rate schedule combinations, including settings such as learning rate, momentum, and weight decay. After training each configuration, I evaluate the results using accuracy, loss, convergence behavior, and training stability. I then compare the results and justify which optimizer and learning-rate schedule combination worked best for this task.

### 1.2. Task Summary Table

| Task Attribute | Activity Value |
|----------------|---------------|
| AI Task Area   | Computer Vision |
| Problem Type   | Multiclass Image Classification |
| Learning Type  | Supervised Transfer Learning / Fine-Tuning |
| Data Modality  | Images |
| Domain         | AI / Machine Learning Engineering Experimentation |
| Model Output   | Predicted image class label |
| Primary Metric | Accuracy, supported by loss and convergence behavior |
| Minimum Target | Not specified by the activity requirements |


## 2. Dataset

### 2.0. Select the Dataset

#### 2.1. Understand the Dataset Options

The `keras.datasets` module provides convenient access to several popular datasets. For this activity, review the pages for these image datasets. Each dataset has its own characteristics, such as the type of data, number of classes, and input shapes. 

- CIFAR10 small images classification dataset
  - [CIFAR10 Keras page](https://keras.io/api/datasets/cifar10/)
  - [CIFAR10 Homepage](https://www.cs.toronto.edu/~kriz/cifar.html)

- Fashion MNIST dataset
  - [Fashion MNIST Keras page](https://keras.io/api/datasets/fashion_mnist/)
  - [Fashion MNIST GitHub repository](https://github.com/zalandoresearch/fashion-mnist)

The starter code below provides a mapping of dataset names to their Keras loader functions and expected input shapes. 

In [4]:
# Map each dataset choice to its Keras loader and expected shapes.
DATASETS = {
    "cifar10": {
        "loader": keras.datasets.cifar10.load_data,
        "expected_shapes": {
            "x_train": (50000, 32, 32, 3),
            "y_train": (50000, 1),
            "x_test": (10000, 32, 32, 3),
            "y_test": (10000, 1),
        },
    },
    "fashion_mnist": {
        "loader": keras.datasets.fashion_mnist.load_data,
        "expected_shapes": {
            "x_train": (60000, 28, 28),
            "y_train": (60000,),
            "x_test": (10000, 28, 28),
            "y_test": (10000,),
        },
    },
}


#### 2.2 Select a Dataset

I have chosen the cifar10 database.


In [5]:
DATASET = "cifar10"
# DATASET = "fashion_mnist"

# Validate the dataset choice before loading the data.
if DATASET not in DATASETS:
    available_datasets = ", ".join(
        f"'{dataset_name}'" for dataset_name in DATASETS
    )

    raise ValueError(
        f"Unknown dataset: {DATASET}. "
        f"Use one of the following: {available_datasets}."
    )



#### 2.3. Load and Split the Selected Dataset

In [17]:
# Load the selected dataset and split it into train and test sets.
dataset_config = DATASETS[DATASET]
(x_train, y_train), (x_test, y_test) = dataset_config["loader"]()


#### 2.4. Verify the Loaded Dataset

In [18]:
# Store the loaded arrays in a dictionary so their shapes can be checked.
dataset_arrays = {
    "x_train": x_train,
    "y_train": y_train,
    "x_test": x_test,
    "y_test": y_test,
}

# Validate the loaded dataset shapes.
for array_name, expected_shape in dataset_config["expected_shapes"].items():
    actual_shape = dataset_arrays[array_name].shape

    assert actual_shape == expected_shape, (
        f"Expected {array_name} shape {expected_shape}, "
        f"but got {actual_shape}. Try reloading the dataset or "
        "check the Keras documentation for the expected shapes."
    )

# Show the selected dataset and array shapes.
print(f"Selected dataset: {DATASET}")
print("Dataset shapes:")

for array_name, array_values in dataset_arrays.items():
    print(f"{array_name}: {array_values.shape}")



Selected dataset: cifar10
Dataset shapes:
x_train: (50000, 32, 32, 3)
y_train: (50000, 1)
x_test: (10000, 32, 32, 3)
y_test: (10000, 1)


### 2.5 Prepare the Training Dataset

The selected dataset is already prepared for this activity because it comes from Keras as a structured benchmark dataset. The data is already separated into training and test sets, and each image already has a matching class label. Because of this, you do not need to collect raw files, clean missing values, merge data sources, or manually create labels for this activity. In later assignments with real-world raw datasets, you may need to prepare the training dataset before the model training process.


---

## Start of Rubric Graded Items

---

## 3. Model

### 3.1 Select a Pre-Trained Model

For this fine tuning task, I selected ResNet50 from Keras Applications because it is good at finding important features in images and has an efficient structure. I will configure it to use the CIFAR-10 input size of 32x32x3 pixels. ResNet50 was originally trained on the large ImageNet dataset using 224x224x3 pixel images. By adapting it for CIFAR-10, I can use features the model has already learned, which helps my model train faster. Its residual connection architecture also helps reduce the vanishing gradient problem. With about 25.6 million parameters, I found that ResNet50 provides a good balance between learning complex patterns and the amount of computing power needed for transfer learning.


Keras Applications Documentation. ResNet50 and ResNet50V2

https://keras.io/api/applications/

In [19]:
from tensorflow.keras.applications import ResNet50

### 3.2 Select and Configure Optimizers

I selected Adam and AdamW because they use different ways to update my pre-trained ResNet50 model. For Adam, I selected a learning rate of 0.001, momentum of 0.9, and weight decay of 0.0001. For AdamW, I selected a learning rate of 0.0001, a beta_1 value of 0.9, and weight decay of None. AdamW does not have a setting called momentum, so I used beta_1 because it works in a similar way by using past gradients to help guide updates. I chose these learning rates so the model makes smaller changes to the weights it already learned. This can help the model learn from the CIFAR-10 dataset without changing the useful features too quickly. Momentum can make training more stable, while weight decay can help prevent overfitting. I will test both optimizers with learning rate schedules later to see which one works better for my model.


https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam

https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/AdamW

In [20]:
# Example structure only. Replace these placeholders with your selected optimizers
# and values. Add or remove dictionary keys as needed for your optimizers.
optimizer_configs = [
    {
        "name": "Adam",
        "learning_rate": 0.001,
        "momentum": 0.9,
        "weight_decay": None,
        "notes": " This is an stochastic gradient descent optimizer.",
    },
    {
        "name": "AdamW",
        "learning_rate": 0.001,
        "momentum": 0.99, # beta_1
        "weight_decay": 0.004,
        "notes": "Optimizer with little memory required.",
    },
]



### 3.3 Apply a Learning Rate Schedule

To choose the learning rate schedules for fine-tuning ResNet50 on CIFAR-10 using Adam and AdamW, I will test two options: Inverse Time Decay with an initial learning rate of 0.001, decay steps of 1000, a decay rate of 0.5, and staircase set to False, and Cosine Decay with an initial learning rate of 0.001, decay steps of 10000, and alpha set to 0.0. I will use these schedules because they will gradually lower the learning rate and help prevent large updates from damaging the features ResNet50 learned from ImageNet. With Inverse Time Decay, I will reduce the learning rate as the training steps increase. The decay steps and decay rate will control how quickly it decreases, while staircase set to False will allow the changes to happen smoothly. This approach will help the model learn quickly at first and make smaller, more stable updates later, reducing the risk of moving past the best weights. With Cosine Decay, I will lower the learning rate smoothly along a cosine curve over 10000 steps until it reaches the minimum level set by alpha at 0.0. This approach will help the model make smaller updates near the end of training, giving it a better chance to find the right weights.


In [21]:
    # Example structure only. Replace these placeholders with your selected schedules
    # and values. Add or remove dictionary keys as needed for your schedules.
    schedule_configs = [
        {
            "name": "inverse_time_decay",
            "initial_learning_rate": 0.001,
            "parameters": {
                "decay_steps": 1000,
                "decay_rate": 0.5,
                "staircase": False 
            },
            "notes": "Improves stability during later fine tuning stages.",
        },
        {
            "name": "cosine_decay",
            "initial_learning_rate": 0.001,
            "parameters": {
                "decay_steps": 10000,
                "alpha": 0.0
            },
            "notes": "Helps the adaptive optimizers settle into optimal local minima.",
        },
    ]



### 3.4 Train the Model

I will use the pre-trained ResNet50 model, which was originally trained on ImageNet, and adjust its input layer to accept the 32 x 32 x 3 image size used by the CIFAR-10 dataset. I will normalize the input pixel values to match the scale expected by the pre-trained model. For this test, I will compile the model using the AdamW optimizer with a learning rate of 0.001, beta_1 of 0.99, and a weight decay of 0.004. I will also use the Cosine Decay learning rate schedule with 10,000 decay steps and alpha set to 0.0. I will train the model using sparse categorical crossentropy loss and measure its accuracy over 5 epochs with a batch size of 64. I will use the training results to check whether the model learns properly and remains stable. I will look for the loss to decrease and the accuracy to increase across the epochs. These results will help me confirm that the data pipeline, model architecture, optimizer, and learning rate schedule work together correctly without exploding gradients or unstable changes during training.


In [22]:
from tensorflow.keras.applications.resnet import preprocess_input

# Full dataset training
X_train_sub = preprocess_input(x_train[:50000].astype("float32"))
X_test_sub = preprocess_input(x_test[:10000].astype("float32"))
y_train_sub = y_train[:50000]
y_test_sub = y_test[:10000]

# Initialize feature extractor with frozen weights for stable transfer learning
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation="softmax")
])

# Configure learning rate schedule and optimizer from predefined configurations
sch_cfg = schedule_configs[1]  # cosine_decay
lr_schedule = CosineDecay(
    initial_learning_rate=sch_cfg["initial_learning_rate"],
    decay_steps=sch_cfg["parameters"]["decay_steps"],
    alpha=sch_cfg["parameters"]["alpha"]
)

opt_cfg = optimizer_configs[1]  # AdamW
optimizer = AdamW(
    learning_rate=lr_schedule,
    beta_1=opt_cfg["momentum"],
    weight_decay=opt_cfg["weight_decay"]
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Execute training loop on the full dataset
history = model.fit(
    X_train_sub, y_train_sub,
    validation_data=(X_test_sub, y_test_sub),
    epochs=10,
    batch_size=64
)


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 107s 129ms/step - accuracy: 0.5769 - loss: 1.6387 - val_accuracy: 0.6210 - val_loss: 1.2908
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 91s 117ms/step - accuracy: 0.6541 - loss: 1.1067 - val_accuracy: 0.6270 - val_loss: 1.2254
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 93s 118ms/step - accuracy: 0.6830 - loss: 0.9715 - val_accuracy: 0.6470 - val_loss: 1.1595
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 141s 117ms/step - accuracy: 0.6956 - loss: 0.9166 - val_accuracy: 0.6506 - val_loss: 1.1664
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 91s 117ms/step - accuracy: 0.7059 - loss: 0.8742 - val_accuracy: 0.6413 - val_loss: 1.1671
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 92s 117ms/step - accuracy: 0.7166 - loss: 0.8337 - val_accuracy: 0.6448 - val_loss: 1.1536
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 92s 117ms/step - accuracy: 0.7286 - loss: 0.7862 - val_accuracy: 0.6566 - val_loss: 1.1052
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 98s 125ms/step - accuracy: 0.7395 - loss:

In [ ]:
print(f"Final Training Accuracy     : {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy   : {history.history['val_accuracy'][-1]:.4f}")
print(f"Final Training Loss         : {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss       : {history.history['val_loss'][-1]:.4f}")

### 3.5 Test Combinations

I will use the pre-trained ResNet50 model, which was originally trained on ImageNet, and adjust its input layer to accept the 32 x 32 x 3 image size used by the CIFAR-10 dataset. I will normalize the input pixel values to match the scale expected by the pre-trained model. For this test, I will compile the model using the AdamW optimizer with a learning rate of 0.001, beta_1 of 0.99, and a weight decay of 0.004. I will also use the Cosine Decay learning rate schedule with 10,000 decay steps and alpha set to 0.0. I will train the model using sparse categorical crossentropy loss and measure its accuracy over 5 epochs with a batch size of 64. I will use the training results to check whether the model learns properly and remains stable. I will look for the loss to decrease and the accuracy to increase across the epochs. These results will help me confirm that the data pipeline, model architecture, optimizer, and learning rate schedule work together correctly without exploding gradients or unstable changes during training.


In [26]:
# TODO (Student): Train and record multiple optimizer/schedule combinations.
# Use the optimizer experiment space from Section 3.2 and the learning rate
# schedule experiment space from Section 3.3. Keep the rest of the training
# setup consistent so the results can be compared fairly in Section 4.1.
# Delete this TODO comment block when done.

# Example result container. Replace or extend this structure as needed.
#experiment_results = []

# List to store performance results for comparison
results = []

# Loop through all combinations of optimizers and schedules
for opt_cfg, sch_cfg in itertools.product(optimizer_configs, schedule_configs):
    print(f"\nTraining with Optimizer: {opt_cfg['name']} | Schedule: {sch_cfg['name']}")
    
    # Learning rate schedule
    if sch_cfg["name"] == "inverse_time_decay":
        lr_schedule = InverseTimeDecay(
            initial_learning_rate=sch_cfg["initial_learning_rate"],
            decay_steps=sch_cfg["parameters"]["decay_steps"],
            decay_rate=sch_cfg["parameters"]["decay_rate"],
            staircase=sch_cfg["parameters"]["staircase"]
        )
    elif sch_cfg["name"] == "cosine_decay":
        lr_schedule = CosineDecay(
            initial_learning_rate=sch_cfg["initial_learning_rate"],
            decay_steps=sch_cfg["parameters"]["decay_steps"],
            alpha=sch_cfg["parameters"]["alpha"]
        )
        
    # Optimizer
    if opt_cfg["name"] == "Adam":
        optimizer = Adam(
            learning_rate=lr_schedule,
            beta_1=opt_cfg["momentum"]
        )
    elif opt_cfg["name"] == "AdamW":
        optimizer = AdamW(
            learning_rate=lr_schedule,
            beta_1=opt_cfg["momentum"],
            weight_decay=opt_cfg["weight_decay"]
        )

    # Build model with frozen ResNet50 base
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(32, 32, 3))
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation="softmax")
    ])

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Train model
    history = model.fit(
        X_train_sub, y_train_sub,
        validation_data=(X_test_sub, y_test_sub),
        epochs=5,
        batch_size=64,
        verbose=1
    )

    # Record final metrics
    results.append({
        "Optimizer": opt_cfg["name"],
        "Schedule": sch_cfg["name"],
        "Final Train Accuracy": history.history['accuracy'][-1],
        "Final Val Accuracy": history.history['val_accuracy'][-1],
        "Final Train Loss": history.history['loss'][-1],
        "Final Val Loss": history.history['val_loss'][-1]
    })

#  Dataframe results
results_df = pd.DataFrame(results)
display(results_df)



Training with Optimizer: Adam | Schedule: inverse_time_decay
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 97s 116ms/step - accuracy: 0.5725 - loss: 1.4922 - val_accuracy: 0.6160 - val_loss: 1.2504
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 90s 115ms/step - accuracy: 0.6599 - loss: 1.0363 - val_accuracy: 0.6384 - val_loss: 1.1229
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 91s 117ms/step - accuracy: 0.6894 - loss: 0.9182 - val_accuracy: 0.6510 - val_loss: 1.0926
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 140s 115ms/step - accuracy: 0.7067 - loss: 0.8567 - val_accuracy: 0.6553 - val_loss: 1.0657
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 91s 116ms/step - accuracy: 0.7176 - loss: 0.8138 - val_accuracy: 0.6614 - val_loss: 1.0373

Training with Optimizer: Adam | Schedule: cosine_decay
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 100s 120ms/step - accuracy: 0.5754 - loss: 1.4997 - val_accuracy: 0.6209 - val_loss: 1.2537
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 93s 119ms/step - accuracy: 0.6494 - loss: 1.1036 - val_accura

,Optimizer,Schedule,Final Train Accuracy,Final Val Accuracy,Final Train Loss,Final Val Loss
0,Adam,inverse_time_decay,0.7176,0.6614,0.8138,1.0373
1,Adam,cosine_decay,0.7022,0.6460,0.8831,1.1458
2,AdamW,inverse_time_decay,0.7174,0.6549,0.8152,1.0442
3,AdamW,cosine_decay,0.7066,0.6473,0.8775,1.1573


# 4. Results

### 4.1 Compare Results

The four combinations of optimizers and learning rate schedules had different results after five training epochs. At the start, all four had similar results, with training accuracy between 0.5725 and 0.5754 and training loss between 1.4922 and 1.6923. The starting validation accuracy was also similar, ranging from 0.6160 to 0.6233. However, the results changed as the models continued training. When I used Adam with inverse time decay, the model reached a final training accuracy of 0.7176, a highest validation accuracy of 0.6614, a training loss of 0.8138, and a validation loss of 1.0373. When I used Adam with cosine decay, the final training accuracy was 0.7022, validation accuracy was 0.6460, training loss was 0.8831, and validation loss was 1.1458. With AdamW and inverse time decay, the model reached a training accuracy of 0.7174, a validation accuracy of 0.6549, a training loss of 0.8152, and a validation loss of 1.0442. Finally, AdamW with cosine decay reached a training accuracy of 0.7066, a validation accuracy of 0.6473, a training loss of 0.8775, and a validation loss of 1.1573.

Based on these results, I found that inverse time decay worked better than cosine decay with both optimizers. It gave the models higher accuracy and lower loss after five epochs. Adam and AdamW also performed almost the same when using the same learning rate schedule. This suggests that the learning rate schedule made a bigger difference than using weight decay in these tests. All four runs showed steady learning, and there were no major signs of overfitting during the five epochs. Keeping the pre-trained feature extractor frozen also helped protect the features ResNet50 had already learned, while allowing the dense classification layer to learn how to sort the CIFAR-10 images into the correct categories.

### 4.2 Justify the Best Combination

The standard Adam optimizer with the inverse time decay schedule was the best choice for fine-tuning ResNet50 on the CIFAR-10 dataset. This setup had the highest validation accuracy of 0.6614 and the lowest validation loss of 1.0373 out of all four tests. It also had a final training accuracy of 0.7176 and a training loss of 0.8138. When I used Adam with cosine decay, the validation accuracy dropped to 0.6460 and the validation loss increased to 1.1458. The training accuracy was also lower at 0.7022, and the training loss was higher at 0.8831. AdamW with inverse time decay came close, with a validation accuracy of 0.6549 and a validation loss of 1.0442. However, standard Adam still performed slightly better on the validation data.

The results show that inverse time decay worked better than cosine decay in both tests. It helped the model keep learning throughout the five epochs instead of slowing down too much too early. I also found that Adam and AdamW had very similar results when using the same learning rate schedule. Because of this, standard Adam was the better choice for this experiment, since AdamW's extra weight decay did not provide a clear improvement. Choosing the right settings is important for helping deep learning models learn well and work with new data.

## References


Keras Applications Documentation. ResNet50 and ResNet50V2

https://keras.io/api/applications/

<div style="padding-left: 0.5in; text-indent: -0.5in;">

Kapoor, A., Gullì, A., & Pal, S. (2022). Deep learning with TensorFlow and Keras: Build and deploy supervised, unsupervised, deep, and reinforcement learning models (3rd ed) [EPUB]. Packt Publishing.

</div>